# Synthetic LRC-ORC Heatmap Replotting

This notebook reads the recovered summary CSV files and recreates the heatmaps without rerunning synthetic graph generation, pruning, or community detection.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repository_root():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        input_dir = (
            candidate
            / "notebook_output_data"
            / "synthetic_module_verification"
        )
        if input_dir.is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find notebook_output_data/synthetic_module_verification."
    )


REPOSITORY_ROOT = find_repository_root()
INPUT_DIR = (
    REPOSITORY_ROOT
    / "notebook_output_data"
    / "synthetic_module_verification"
)
OUTPUT_DIR = INPUT_DIR / "replotted_heatmaps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading summary CSVs from: {INPUT_DIR}")
print(f"Saving replotted figures to: {OUTPUT_DIR}")

In [ ]:
REQUIRED_COLUMNS = {
    "lrc_pct",
    "orc_pct",
    "ARI_mean",
    "NMI_mean",
    "Runtime_mean",
}

summary_files = sorted(
    INPUT_DIR.glob("*_summary.csv")
)

summary_tables = {}
skipped_files = {}

for csv_path in summary_files:
    dataframe = pd.read_csv(csv_path)
    missing_columns = REQUIRED_COLUMNS - set(dataframe.columns)

    if missing_columns:
        skipped_files[csv_path.name] = sorted(missing_columns)
        continue

    summary_tables[csv_path.stem] = dataframe

print(f"Compatible summary files: {len(summary_tables)}")
for name, dataframe in summary_tables.items():
    print(f"  {name}: {len(dataframe)} rows")

if skipped_files:
    print("Skipped non-heatmap summaries:")
    for name, missing_columns in skipped_files.items():
        print(f"  {name}: missing {missing_columns}")

if not summary_tables:
    raise ValueError("No compatible LRC-ORC summary CSV files were found.")

In [ ]:
def format_label(value):
    if float(value).is_integer():
        return str(int(value))
    return f"{value:g}"


def plot_heatmap(dataframe, value_column, title, output_path, color_map):
    matrix = (
        dataframe
        .pivot_table(
            index="lrc_pct",
            columns="orc_pct",
            values=value_column,
            aggfunc="mean",
        )
        .sort_index()
        .sort_index(axis=1)
    )

    figure, axis = plt.subplots(figsize=(9, 7))
    image = axis.imshow(
        matrix.values,
        aspect="auto",
        cmap=color_map,
    )

    axis.set_xticks(np.arange(len(matrix.columns)))
    axis.set_yticks(np.arange(len(matrix.index)))
    axis.set_xticklabels([format_label(value) for value in matrix.columns])
    axis.set_yticklabels([format_label(value) for value in matrix.index])
    axis.set_xlabel("ORC pruning ratio (%)")
    axis.set_ylabel("LRC pruning ratio (%)")
    axis.set_title(title)

    finite_values = matrix.to_numpy(dtype=float)
    midpoint = np.nanmean(finite_values) if np.isfinite(finite_values).any() else 0.0

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix.iloc[row_index, column_index]
            if pd.isna(value):
                label = "missing"
            elif value_column == "Runtime_mean":
                label = f"{value:.2f}"
            else:
                label = f"{value:.3f}"

            text_color = "white" if value >= midpoint else "black"
            axis.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center",
                color=text_color,
                fontsize=8,
            )

    colorbar = figure.colorbar(image, ax=axis)
    colorbar.set_label(value_column.replace("_", " "))
    figure.tight_layout()
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(figure)

    return matrix


plot_specs = [
    ("ARI_mean", "ARI", "viridis"),
    ("NMI_mean", "NMI", "plasma"),
    ("Runtime_mean", "Runtime", "magma"),
]

generated_files = []

for stem, dataframe in summary_tables.items():
    for value_column, label, color_map in plot_specs:
        output_path = OUTPUT_DIR / f"{stem}_{label.lower()}_heatmap.png"
        plot_heatmap(
            dataframe=dataframe,
            value_column=value_column,
            title=f"{stem}: {label}",
            output_path=output_path,
            color_map=color_map,
        )
        generated_files.append(output_path)

print(f"Generated {len(generated_files)} heatmaps.")
for output_path in generated_files:
    print(output_path)